# Task 1: Multi-Class Benchmark Classification Pipeline (Iris Dataset)

**Objective:** Build and compare Logistic Regression and Decision Tree classification models using the Iris dataset.

This notebook performs data loading, exploratory visualization, an 80/20 train-test split, feature scaling, model training, evaluation using accuracy/precision/recall/F1-score, confusion matrices, and a decision-boundary visualization.

## 1. Import the required libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

sns.set_theme(style="whitegrid")

## 2. Load the Iris dataset

In [ ]:
iris = load_iris()
X = iris.data
y = iris.target

df = pd.DataFrame(X, columns=iris.feature_names)
df["species"] = [iris.target_names[i] for i in y]

print("Dataset shape:", df.shape)
display(df.head())
print("\nClass counts:")
print(df["species"].value_counts())

## 3. Exploratory data visualization (pairplot)

In [ ]:
sns.pairplot(df, hue="species", diag_kind="hist")
plt.suptitle("Iris Dataset: Feature Relationships", y=1.02)
plt.show()

## 4. Split the data into training and testing sets (80/20)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

## 5. Feature scaling and model training

A Scikit-learn pipeline applies `StandardScaler` to the training data and then trains Logistic Regression. The Decision Tree is also trained for comparison; decision trees generally do not require feature scaling.

In [ ]:
logistic_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, random_state=42)
)

decision_tree_model = DecisionTreeClassifier(random_state=42)

logistic_model.fit(X_train, y_train)
decision_tree_model.fit(X_train, y_train)

y_pred_lr = logistic_model.predict(X_test)
y_pred_dt = decision_tree_model.predict(X_test)

print("Both models have been trained successfully.")

## 6. Evaluate and compare model performance

In [ ]:
def evaluate_model(model_name, y_true, y_pred):
    return {
        "Model": model_name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision (weighted)": precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "Recall (weighted)": recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "F1-score (weighted)": f1_score(y_true, y_pred, average="weighted", zero_division=0)
    }

results = pd.DataFrame([
    evaluate_model("Logistic Regression", y_test, y_pred_lr),
    evaluate_model("Decision Tree", y_test, y_pred_dt)
])

display(results.round(4))

## 7. Classification reports

In [ ]:
print("LOGISTIC REGRESSION REPORT\n")
print(classification_report(y_test, y_pred_lr, target_names=iris.target_names, zero_division=0))

print("DECISION TREE REPORT\n")
print(classification_report(y_test, y_pred_dt, target_names=iris.target_names, zero_division=0))

## 8. Plot confusion matrices for both models

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, predictions, title in [
    (axes[0], y_pred_lr, "Logistic Regression"),
    (axes[1], y_pred_dt, "Decision Tree")
]:
    cm = confusion_matrix(y_test, predictions)
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues", cbar=False,
        xticklabels=iris.target_names,
        yticklabels=iris.target_names,
        ax=ax
    )
    ax.set_title(title)
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("Actual label")

plt.tight_layout()
plt.show()

## 9. Decision-boundary visualization

For a 2D plot, this section uses the first two Iris features (sepal length and sepal width). Separate models are trained on these two features only so that their decision regions can be displayed.

In [ ]:
X_2d = X[:, :2]
X2_train, X2_test, y2_train, y2_test = train_test_split(
    X_2d, y, test_size=0.20, random_state=42, stratify=y
)

lr_2d = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=42))
dt_2d = DecisionTreeClassifier(random_state=42)
lr_2d.fit(X2_train, y2_train)
dt_2d.fit(X2_train, y2_train)

x_min, x_max = X_2d[:, 0].min() - 0.5, X_2d[:, 0].max() + 0.5
y_min, y_max = X_2d[:, 1].min() - 0.5, X_2d[:, 1].max() + 0.5
xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 300),
    np.linspace(y_min, y_max, 300)
)
grid = np.c_[xx.ravel(), yy.ravel()]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, model, title in [
    (axes[0], lr_2d, "Logistic Regression (2 features)"),
    (axes[1], dt_2d, "Decision Tree (2 features)")
]:
    regions = model.predict(grid).reshape(xx.shape)
    ax.contourf(xx, yy, regions, alpha=0.25, cmap="viridis")
    scatter = ax.scatter(
        X2_test[:, 0], X2_test[:, 1], c=y2_test,
        cmap="viridis", edgecolor="black"
    )
    ax.set_title(title)
    ax.set_xlabel(iris.feature_names[0])
    ax.set_ylabel(iris.feature_names[1])

plt.tight_layout()
plt.show()

## 10. Conclusion

In this task, the Iris dataset was used to build a multi-class classification pipeline. Logistic Regression and Decision Tree models were trained and compared using accuracy, weighted precision, weighted recall, and weighted F1-score. Confusion matrices and decision-boundary plots were also generated to understand model performance. The metric table can be used to identify which model performed better on this test split.